# PSA_CenterGravity: 4 隅の信号の重心(位置)

`PSA_CalcFADC` の 4 隅の積分 $S_0 \sim S_3$(`wave_sum`)から、ブロックの中で光った位置を重心(Anger 方式)で求めます。
`psa_decode.py` と同じ式です。

$$x = \frac{S_0 - S_1 - S_2 + S_3}{S_0 + S_1 + S_2 + S_3}, \qquad y = \frac{-S_0 - S_1 + S_2 + S_3}{S_0 + S_1 + S_2 + S_3}$$

HG の 4 本で `x_hg`, `y_hg`、LG の 4 本で `x_lg`, `y_lg` です。和が 0 のときは 0 にします。
式から、4 隅の位置は $S_0 = (+, -)$、$S_1 = (-, -)$、$S_2 = (-, +)$、$S_3 = (+, +)$ です。

8 × 8 のピクセルの結晶の光が MPPC に広がって届くので、図にすると 64 個の点の集まり(バタフライマップ)が見えます。
重心は端に行くほど縮むので、点の並びは歪んだ格子になります。これをピクセルに分けるのが次の
[5_PSA_Calibration](../5_PSA_Calibration/PSA_Calibration.ipynb) です。

`save_maps: true` のとき、ボード・ch ごとの `x_hg`-`y_hg` の図 `PSA_CenterGravity/Map_haNN_chC` を保存します
(範囲とビンは `grid_spect.py` と同じ: x −0.32〜0.32、y −0.16〜0.16、300 × 300)。

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CACHE = REPO / "tutorials" / "cache"   # チェーンの出力の置き場(チュートリアル間で共有、git には入れない)
CACHE.mkdir(exist_ok=True)

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    a = np.frombuffer(h.GetArray(), dtype=np.float64, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データの準備

2026-11-03 の UVSOR での **HA_cal のラン `20261103_run01`**(PSA だけ、14 ボード、約 840 MB)を使います。
位置の較正(`calib/psa/position_haNN.csv`)を作ったときと同じ日の較正データです。
データは解析用の計算機の D ドライブ(`/mnt/d/UVSOR2026_late`)にあります。
ほかの場所にあるときは、環境変数 `SMILE3_PSA_CAL_DATA` に `UVSOR2026_late` に当たるディレクトリを設定してください。

1 つのランのボードは、2 つの storage に分かれて書かれています。

| storage | ディレクトリ | PSA のボード |
|---|---|---|
| storage_g1 | `killua/<ラン>` | ha00〜ha05 |
| storage_g0 | `calltt/<ラン>` | ha06〜ha16 |

`run_chain.py` には、片方を `--data-dir`、もう片方を `PSA_ReadData.extra_data_dirs` で渡します。

In [ ]:
DATA = Path(os.environ.get("SMILE3_PSA_CAL_DATA", "/mnt/d/UVSOR2026_late"))
RUN = "20261103_run01"
RUN_DIRS = [DATA / "killua" / RUN, DATA / "calltt" / RUN]
for d in RUN_DIRS:
    print("OK     " if d.is_dir() else "MISSING", d)

## 2. チェーンを流す

設定ファイル `smile3_psa`(`PSA_ReadData → PSA_CalcFADC → PSA_CenterGravity → PSA_Calibration → PSA_TreeWriter`)で、ラン全体を流します(40 秒ほど)。
ターミナルでは、リポジトリの `run_script` で次を実行するのと同じです。

```bash
python3 run_chain.py smile3_psa -o psa_20261103_run01.root \
    --data-dir /mnt/d/UVSOR2026_late/killua/20261103_run01 \
    --set 'PSA_ReadData.extra_data_dirs=["/mnt/d/UVSOR2026_late/calltt/20261103_run01"]'
```

出力は `tutorials/cache/psa_cal_<ラン>.root` に置き、既にあれば流し直しません(PSA のチュートリアルと `4_MakeCalibration/PSA_Energy` で共有)。
`calib/psa` の較正ファイルを変えたときは、このファイルを消して流し直してください。

In [ ]:
def run_chain(out, *args):
    """run_chain.py を実行する(出力が既にあれば何もしない)"""
    cmd = [sys.executable, str(RUN_CHAIN), "smile3_psa", "-o", str(out), "--data-dir", str(RUN_DIRS[0]),
           "--set", "PSA_ReadData.extra_data_dirs=" + json.dumps([str(RUN_DIRS[1])]), *args]
    print(" ".join(cmd))
    if not out.exists():
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
        r.check_returncode()

OUT = CACHE / f"psa_cal_{RUN}.root"   # 今の calib/psa(位置・温度ゲイン・エネルギー)で流したもの
run_chain(OUT)

In [ ]:
def read_psa(path, columns):
    """PSA_TreeWriter/psa の列を numpy の配列で読む(short の列は int に直す)"""
    d = ROOT.RDataFrame("PSA_TreeWriter/psa", str(path)).AsNumpy(columns)
    out = {}
    for k, v in d.items():
        if v.dtype == object and len(v) and hasattr(v[0], "__len__"):
            out[k] = np.stack([np.asarray(x) for x in v])   # 配列の列 -> 2 次元
        elif v.dtype == object:
            out[k] = v.astype(int)
        else:
            out[k] = v
    return out

## 3. バタフライマップ

ha03 の 6 ch の `Map_ha03_chC` です。色は対数にしています。
1 ラン(ch あたり約 1.3 万ヒット)を 300 × 300 のビンに入れているので、ほとんどのビンは 0 か 1 で、点の集まりはまだ見えません。

In [ ]:
f = ROOT.TFile(str(OUT))
fig, axs = plt.subplots(2, 3, figsize=(16, 6.5))
for c, ax in enumerate(axs.flat):
    a, ex, ey = th2(f.Get(f"PSA_CenterGravity/Map_ha03_ch{c}"))
    m = ax.pcolormesh(ex, ey, a.T, norm=LogNorm(vmin=1), shading="flat")
    ax.set_title(f"Map_ha03_ch{c}"); ax.set_xlabel("x_hg"); ax.set_ylabel("y_hg"); ax.set_aspect("equal")
    fig.colorbar(m, ax=ax, fraction=0.025)
fig.tight_layout(); plt.show()

ツリーから、ビンを 120 × 60 に粗くし、ピークがトリガーの時刻(26〜29 サンプル)で `hg` が 3 万以上のヒットだけで描くと、
8 列の点の集まりが見えてきます(位置の較正 `make_psa_position.py` も、同じように選んだヒットを使う)。
64 個がはっきり分かれるには、もっと多くのランを足す必要があります。

In [ ]:
t = read_psa(OUT, ["board", "ch", "x_hg", "y_hg", "x_lg", "y_lg", "hg", "peak_id", "satu_h", "satu_l"])
good = (t["peak_id"] >= 26) & (t["peak_id"] <= 29) & (t["hg"] >= 30e3) & (t["satu_h"] == 0) & (t["satu_l"] == 0)
fig, axs = plt.subplots(2, 3, figsize=(16, 6.5))
for c, ax in enumerate(axs.flat):
    s = good & (t["board"] == 3) & (t["ch"] == c)
    h = ax.hist2d(t["x_hg"][s], t["y_hg"][s], bins=[120, 60], range=[[-0.32, 0.32], [-0.16, 0.16]], cmap="viridis")
    ax.set_title(f"ha03 ch{c} ({s.sum()} hits)"); ax.set_xlabel("x_hg"); ax.set_ylabel("y_hg"); ax.set_aspect("equal")
fig.tight_layout(); plt.show()

## 4. 図に入れるヒットの選び方

図には、次をすべて満たすヒットだけを入れます(ツリーの `x_hg`, `y_hg` は全部のヒットにあります)。

| パラメータ | 条件 |
|---|---|
| `map_peak_id_min` | `peak_id` がこれ以上(既定 0 = 選ばない) |
| `map_skip_saturated` | true なら、HG か LG が飽和したヒットを入れない |
| `map_hg_min` | `hg` がこれ以上 |

ツリーから同じ図を作り、`peak_id` で選ぶと何が変わるかを見ます(ha03 ch0、`hg` 3 万以上)。
`peak_id` がずれたヒット(ほかのボードのトリガーで記録した、時刻のずれた信号)は、格子の外や間に散らばります。

In [ ]:
s = (t["board"] == 3) & (t["ch"] == 0) & (t["satu_h"] == 0) & (t["satu_l"] == 0) & (t["hg"] >= 30e3)
rng = [[-0.32, 0.32], [-0.16, 0.16]]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, (name, cut) in zip(ax, [("all", s), ("peak_id in 26-30", s & (t["peak_id"] >= 26) & (t["peak_id"] <= 30)),
                               ("peak_id outside 26-30", s & ((t["peak_id"] < 26) | (t["peak_id"] > 30)))]):
    h = a.hist2d(t["x_hg"][cut], t["y_hg"][cut], bins=[120, 60], range=rng)
    a.set_title(f"ha03 ch0: {name} ({cut.sum()} hits)"); a.set_xlabel("x_hg"); a.set_ylabel("y_hg"); a.set_aspect("equal")
fig.tight_layout(); plt.show()

## 5. HG と LG の重心

`x_lg`, `y_lg` は LG の 4 本で求めた重心です。ゲインが小さい分ばらつきが大きいので、ピクセルを決めるのには `x_hg`, `y_hg` を使っています。

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, (u, v) in zip(ax, [("x_hg", "x_lg"), ("y_hg", "y_lg")]):
    h = a.hist2d(t[u][s], t[v][s], bins=200, range=[[-0.35, 0.35], [-0.35, 0.35]], norm=LogNorm())
    a.set_xlabel(u); a.set_ylabel(v); a.set_title("ha03 ch0"); a.set_aspect("equal")
fig.tight_layout(); plt.show()

## 6. パラメータ

In [ ]:
show_params("smile3_psa", "PSA_CenterGravity")